# Generate frequencies.txt

Generates GTFS frequencies file with headway intervals for each route.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from gtfs_common import map_route_params, route_id_from_trip_id


## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]

# Load headways configured by route
headway_by_route = p["frequencies"].get("headway_by_route", {})
interval_global = headway_by_route.get("default", 13.11)

print(f"Global headway: {interval_global} minutes")
print(f"Routes with specific headway: {len([k for k in headway_by_route.keys() if k != 'default'])}")

params_frequencies = {
    "start_time": p["frequencies"]["start_time"],
    "end_time": p["frequencies"]["end_time"],
    "headway_secs": round(interval_global * 60, 2),
    "exact_times": p["frequencies"]["exact_times"],
}

Global headway: 13.11 minutes
Routes with specific headway: 5


In [4]:
# --- GTFS Folder ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_GTFS.absolute()}")

# --- Processed Folder ---
PATH_DIR_processed = Path(f"../data/{CITY}/processed")
PATH_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_processed.absolute()}")

Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read files

In [ ]:
stop_times = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")
routes_df = pd.read_csv(PATH_DIR_GTFS / "routes.txt")

headway_by_route_id = map_route_params(
    routes_df,
    headway_by_route,
    interval_global,
    kind="headway_min",
)
stop_times.head()


## Generate frequencies.txt

In [8]:
# Basic validations
def _valid_hhmmss(s):
    try:
        h,m,sec = map(int, str(s).split(":"))
        return (h >= 0 and 0 <= m < 60 and 0 <= sec < 60)
    except Exception:
        return False


good_start_bool = _valid_hhmmss(params_frequencies["start_time"])
good_end_bool   = _valid_hhmmss(params_frequencies["end_time"])

# If 'not' start_time IS VALID 'or' 'not' end_time IS VALID:
if not (good_start_bool and good_end_bool):
    raise ValueError("Invalid time format in start_time/end_time (use HH:MM:SS; hours >=24 allowed).")

In [ ]:
def build_frequencies_by_route(stop_times, headway_by_route_id, global_params):
    trips_unique = stop_times[["trip_id"]].drop_duplicates().copy()
    trips_unique["route_id"] = trips_unique["trip_id"].map(route_id_from_trip_id)
    missing = sorted(set(trips_unique["route_id"]) - set(headway_by_route_id))
    if missing:
        raise ValueError(f"No headway for routes: {missing}")

    trips_unique["headway_secs"] = (
        trips_unique["route_id"].map(headway_by_route_id) * 60
    ).round().astype(int)
    trips_unique["start_time"] = global_params["start_time"]
    trips_unique["end_time"] = global_params["end_time"]
    trips_unique["exact_times"] = int(global_params["exact_times"])
    cols = ["trip_id", "start_time", "end_time", "headway_secs", "exact_times"]
    return trips_unique[cols]


In [ ]:
global_params = {
    "start_time": p["frequencies"]["start_time"],
    "end_time": p["frequencies"]["end_time"],
    "exact_times": p["frequencies"]["exact_times"],
}

frequencies = build_frequencies_by_route(stop_times, headway_by_route_id, global_params)

headway_summary = frequencies.copy()
headway_summary["route_id"] = headway_summary["trip_id"].map(route_id_from_trip_id)
headway_summary = headway_summary.groupby("route_id")["headway_secs"].first().reset_index()
headway_summary["headway_min"] = (headway_summary["headway_secs"] / 60).round(2)
print(headway_summary.head(10))
frequencies.head()


## Export

In [11]:
# Export frequencies to GTFS format
frequencies.to_csv(PATH_DIR_GTFS / "frequencies.txt", index=False)
print(f"✓ Exported {len(frequencies)} frequency records to frequencies.txt")

✓ Exported 56 frequency records to frequencies.txt
